In [1]:
import csv
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

FAQ_FILE = ROOT / "rag" / "hotel_faq_gold.csv"
KB_FILE = ROOT / "knowledge_base" / "cancellation_process.txt"

KB_OLD = "1. Provide Reservation Details\r\n\r\nThe guest gives the booking reference and the last name on the booking."
KB_NEW = "1. Find the Booking\r\n\r\nThe guest logs in and chooses the booking from their account. A booking made by phone, by email, through a travel agent or on a booking platform is first added to the account with its booking reference and the last name on it (see Reservation Security)."

REPLACE = {
    "The cancellation process: give the booking reference and last name; customer service checks the rate plan and the 48-hour deadline, tells you any charge before cancelling, cancels the booking, processes any refund within 3 business days, and emails you a cancellation confirmation.":
    "The cancellation process: log in and choose the booking from your account (a booking made by phone, through an agency or on a booking platform is first added with its reference and last name); the rate plan and the 48-hour deadline are checked, you are told any charge before cancelling, the booking is cancelled, any refund is processed within 3 business days, and a cancellation confirmation is emailed to you.",

    "خطوات الإلغاء: تقديم رقم الحجز واسم العائلة، ثم تتحقق خدمة العملاء من نوع السعر ومهلة الـ 48 ساعة، وتوضح لك أي رسوم قبل الإلغاء، ثم يُلغى الحجز، ويُعالج أي استرداد خلال 3 أيام عمل، ويصلك تأكيد الإلغاء بالبريد الإلكتروني.":
    "خطوات الإلغاء: تسجيل الدخول واختيار الحجز من حسابك (ويُضاف الحجز الذي تم بالهاتف أو عبر وكالة أو منصة حجز أولاً برقمه واسم العائلة)، ثم يُتحقق من نوع السعر ومهلة الـ 48 ساعة، وتوضَّح لك أي رسوم قبل الإلغاء، ثم يُلغى الحجز، ويُعالج أي استرداد خلال 3 أيام عمل، ويصلك تأكيد الإلغاء بالبريد الإلكتروني.",

    "Pour annuler : vous donnez le numéro de réservation et le nom de famille ; le service client vérifie le tarif et le délai de 48 heures, vous indique les éventuels frais avant d'annuler, annule la réservation, traite le remboursement sous 3 jours ouvrés et vous envoie une confirmation par e-mail.":
    "Pour annuler : connectez-vous et choisissez la réservation dans votre compte (une réservation faite par téléphone, par une agence ou sur une plateforme est d'abord ajoutée avec sa référence et le nom de famille) ; le tarif et le délai de 48 heures sont vérifiés, les éventuels frais vous sont indiqués avant d'annuler, la réservation est annulée, le remboursement est traité sous 3 jours ouvrés et une confirmation vous est envoyée par e-mail.",

    "Порядок отмены: вы сообщаете номер брони и фамилию; служба поддержки проверяет тариф и срок 48 часов, заранее называет возможную плату, отменяет бронь, оформляет возврат в течение 3 рабочих дней и присылает подтверждение отмены по электронной почте.":
    "Порядок отмены: войдите в аккаунт и выберите бронь (бронь, сделанную по телефону, через агентство или на платформе, сначала добавляют по её номеру и фамилии); проверяются тариф и срок 48 часов, заранее называется возможная плата, бронь отменяется, возврат оформляется в течение 3 рабочих дней, а подтверждение отмены приходит по электронной почте.",

    "بسيطة يا فندم: قولي رقم الحجز والاسم، وأنا أشوف نوع حجزك وفاضل قد إيه على الـ 48 ساعة، وأقولك لو في رسوم قبل ما ألغي. بعد كده بلغيه، والاسترداد لو في بيتعمل خلال 3 أيام عمل، وبيوصلك تأكيد على الإيميل.":
    "بسيطة يا فندم: سجّل دخولك واختار الحجز من حسابك، وأنا أشوف نوع حجزك وفاضل قد إيه على الـ 48 ساعة، وأقولك لو في رسوم قبل ما ألغي. بعد كده بلغيه، والاسترداد لو في بيتعمل خلال 3 أيام عمل، وبيوصلك تأكيد على الإيميل. ولو الحجز معمول بالتليفون أو من شركة سياحة، ضيفه لحسابك الأول برقمه واسم العيلة.",

    "يمكن طلب الإلغاء عبر خدمة العملاء مع رقم الحجز واسم العائلة. في السعر المرن يكون الإلغاء مجانياً حتى 48 ساعة قبل الوصول وبعدها تُحتسب الليلة الأولى، ولا يُرد الحجز غير القابل للاسترداد.":
    "يمكن إلغاء الحجز بعد تسجيل الدخول واختياره من حسابك. في السعر المرن يكون الإلغاء مجانياً حتى 48 ساعة قبل الوصول وبعدها تُحتسب الليلة الأولى، ولا يُرد الحجز غير القابل للاسترداد.",

    "تمام، ابعتلي رقم الحجز والاسم اللي متسجل بيه. لو الحجز مرن والإلغاء قبل الوصول بـ 48 ساعة يبقى ببلاش، ولو بعد كده بتتحسب أول ليلة، والحجز اللي مش بيترد مش بيرجع فلوسه.":
    "تمام، سجّل دخولك واختار الحجز من حسابك. لو الحجز مرن والإلغاء قبل الوصول بـ 48 ساعة يبقى ببلاش، ولو بعد كده بتتحسب أول ليلة، والحجز اللي مش بيترد مش بيرجع فلوسه.",

    "يمكنني المساعدة. يرجى تزويدي برقم الحجز والاسم المسجل به حتى أتمكن من تحديد الحجز الصحيح.":
    "يمكنني المساعدة. يرجى تسجيل الدخول واختيار الحجز من حسابك حتى أتمكن من تحديده.",

    "تمام أقدر أساعدك. قوللي رقم الحجز والاسم اللي متسجل بيه عشان ألاقيه.":
    "تمام أقدر أساعدك. سجّل دخولك وقوللي أنهي حجز من حجوزاتك عشان ألاقيه.",

    "يمكنني المساعدة. ما الذي تريد تغييره — التواريخ أم نوع الغرفة أم عدد النزلاء؟ وهل تعطيني رقم الحجز حتى أجد الحجز؟":
    "يمكنني المساعدة. ما الذي تريد تغييره — التواريخ أم نوع الغرفة أم عدد النزلاء؟ وأي حجز من الحجوزات الموجودة في حسابك؟",

    "أقدر أساعدك. عايز تغير إيه بالظبط: التواريخ ولا نوع الأوضة ولا عدد الأفراد؟ وابعتلي رقم الحجز عشان ألاقيه.":
    "أقدر أساعدك. عايز تغير إيه بالظبط: التواريخ ولا نوع الأوضة ولا عدد الأفراد؟ وقوللي أنهي حجز من اللي في حسابك.",

    "يمكنني المساعدة في ذلك. هل تعطيني رقم الحجز والاسم المسجل به حتى أجد الحجز الصحيح؟ شروط الإلغاء تعتمد على الحجز تحديداً.":
    "يمكنني المساعدة في ذلك. أي حجز من الحجوزات الموجودة في حسابك؟ شروط الإلغاء تعتمد على الحجز تحديداً.",

    "أقدر أساعدك. ابعتلي رقم الحجز والاسم اللي متسجل بيه عشان ألاقي الحجز الصح، وشروط الإلغاء بتعتمد على الحجز نفسه.":
    "أقدر أساعدك. قوللي أنهي حجز من اللي في حسابك عشان ألاقي الحجز الصح، وشروط الإلغاء بتعتمد على الحجز نفسه.",

    "قدّم رقم الحجز ويمكن الاطلاع على حالة الدفع المرتبطة به. تُسجَّل حالة الدفع كمعلقة أو مدفوعة أو فاشلة أو مستردة أو مستردة جزئياً، ويمكن لخدمة العملاء توضيح ما ينطبق على حجزك.":
    "اختر الحجز من حسابك ويمكن الاطلاع على حالة الدفع المرتبطة به. تُسجَّل حالة الدفع كمعلقة أو مدفوعة أو فاشلة أو مستردة أو مستردة جزئياً، ويمكن لخدمة العملاء توضيح ما ينطبق على حجزك.",

    "ابعتلي رقم الحجز وأشوفلك حالة الدفع: معلق ولا مدفوع ولا فشل ولا اترد كله أو جزء منه.":
    "قوللي أنهي حجز من اللي في حسابك وأشوفلك حالة الدفع: معلق ولا مدفوع ولا فشل ولا اترد كله أو جزء منه.",

    "المبلغ المتبقي هو إجمالي الحجز ناقص المبلغ المدفوع بالفعل. قدّم رقم الحجز ويمكن قراءة المبلغ المدفوع والرصيد المستحق من سجل الحجز.":
    "المبلغ المتبقي هو إجمالي الحجز ناقص المبلغ المدفوع بالفعل. اختر الحجز من حسابك ويمكن قراءة المبلغ المدفوع والرصيد المستحق من سجل الحجز.",

    "الباقي هو إجمالي الحجز ناقص اللي دفعته، ابعتلي رقم الحجز وأقولك دفعت كام وفاضل عليك كام.":
    "الباقي هو إجمالي الحجز ناقص اللي دفعته، قوللي أنهي حجز من اللي في حسابك وأقولك دفعت كام وفاضل عليك كام.",

    "سأتحقق من هذا الكود في نظام الفندق الآن. يُرجى تزويدي برقم الحجز أو تواريخ الإقامة لتطبيق الخصم إذا كان الكود سارياً.":
    "سأتحقق من هذا الكود في نظام الفندق الآن. أي حجز من حسابك تريد تطبيق الخصم عليه، أو ما تواريخ الإقامة إذا كان حجزاً جديداً؟",

    "هتأكدلك من الكود في السيستم دلوقتي. ابعتلي رقم الحجز أو تواريخ الإقامة عشان نطبق الخصم لو الكود شغال.":
    "هتأكدلك من الكود في السيستم دلوقتي. قوللي أنهي حجز من اللي في حسابك، أو تواريخ الإقامة لو حجز جديد، عشان نطبق الخصم لو الكود شغال.",

    "تعتمد الرسوم على نوع السعر وتوقيت الإلغاء: الحجز المرن الملغى قبل أقل من 48 ساعة من الوصول تُحتسب فيه الليلة الأولى، ولا تُرد الحجوزات غير القابلة للاسترداد. وإذا لم تتطابق الرسوم مع حجزك، يُرجى تزويد خدمة العملاء برقم الحجز لمراجعتها كشكوى فوترة.":
    "تعتمد الرسوم على نوع السعر وتوقيت الإلغاء: الحجز المرن الملغى قبل أقل من 48 ساعة من الوصول تُحتسب فيه الليلة الأولى، ولا تُرد الحجوزات غير القابلة للاسترداد. وإذا لم تتطابق الرسوم مع حجزك، يُرجى تحديد الحجز من حسابك لمراجعتها كشكوى فوترة.",

    "خليني أراجع معاك. لو حجزك مرن ولغيته قبل الوصول بأقل من 48 ساعة بتتحسب أول ليلة، ولو قبلها مفيش رسوم. ابعتلي رقم الحجز وأشوف، ولو الخصم طلع غلط هسجلها شكوى على الفاتورة.":
    "خليني أراجع معاك. لو حجزك مرن ولغيته قبل الوصول بأقل من 48 ساعة بتتحسب أول ليلة، ولو قبلها مفيش رسوم. قوللي أنهي حجز من اللي في حسابك وأشوف، ولو الخصم طلع غلط هسجلها شكوى على الفاتورة.",
}

kb_text = KB_FILE.read_bytes().decode("utf-8")
if KB_OLD not in kb_text:
    KB_OLD = KB_OLD.replace("\r\n", "\n")
    KB_NEW = KB_NEW.replace("\r\n", "\n")
print("KB section found:", KB_OLD in kb_text)
KB_FILE.write_bytes(kb_text.replace(KB_OLD, KB_NEW).encode("utf-8"))

raw = FAQ_FILE.read_bytes()
bom = raw.startswith(b"\xef\xbb\xbf")
newline = "\r\n" if b"\r\n" in raw[:2000] else "\n"

with open(FAQ_FILE, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    fields = reader.fieldnames
    rows = list(reader)

used = {}
for old in REPLACE:
    used[old] = 0

changed = 0
for row in rows:
    for col in ["answer", "answer_egyptian"]:
        if row[col] in REPLACE:
            used[row[col]] += 1
            row[col] = REPLACE[row[col]]
            changed += 1

with open(FAQ_FILE, "w", encoding="utf-8-sig" if bom else "utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields, lineterminator=newline)
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

print("cells changed:", changed)
for old in used:
    if used[old] == 0:
        print("NOT FOUND:", old[:60])

KB section found: True
cells changed: 38
